In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!pip install pytest joblib numpy pandas scikit-learn mgwr

In [7]:
%%writefile test_tourism_models.py

import warnings
import pytest
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------------------------------
# FIXTURE: Load Saved Models & Files
# ---------------------------------------------------
@pytest.fixture(scope="module")
def tourism_assets():
    return {
        "gwr": joblib.load("/content/drive/MyDrive/Data Science Group Project/gwr_model.joblib"),
        "scaler": joblib.load("/content/drive/MyDrive/Data Science Group Project/data_scaler.joblib"),
        "future": pd.read_csv("/content/drive/MyDrive/Data Science Group Project/future_predictions.csv"),
        "regional": pd.read_csv("/content/drive/MyDrive/Data Science Group Project/regional_demand_summary.csv"),
        "linear": joblib.load("/content/drive/MyDrive/Data Science Group Project/linear_regression_model.joblib")
    }

# ---------------------------------------------------
# TEST 1: Model Loading
# ---------------------------------------------------
def test_model_loading(tourism_assets):
    assert tourism_assets["gwr"] is not None
    assert tourism_assets["scaler"] is not None

# ---------------------------------------------------
# TEST 2: Scaler Transformation
# ---------------------------------------------------
def test_scaler_output(tourism_assets):
    scaler = tourism_assets["scaler"]

    # Get expected number of features
    n_features = scaler.n_features_in_

    # Create dummy sample with correct feature count
    sample_input = np.ones((1, n_features))

    scaled = scaler.transform(sample_input)

    assert isinstance(scaled, np.ndarray)
    assert scaled.shape == (1, n_features)

# ---------------------------------------------------
# TEST 3: GWR Model R² Reasonable
# ---------------------------------------------------
def test_gwr_r2(tourism_assets):
    gwr = tourism_assets["gwr"]

    # Check model has R2 attribute
    assert hasattr(gwr, "R2")

    # Ensure R² is within logical bounds
    assert -1 <= gwr.R2 <= 1

# ---------------------------------------------------
# TEST 4: Future Predictions Structure
# ---------------------------------------------------
def test_future_predictions_structure(tourism_assets):
    future = tourism_assets["future"]

    assert "district" in future.columns
    assert "year" in future.columns
    assert "predicted_tourism" in future.columns

    assert future["year"].min() >= 2027

# ---------------------------------------------------
# TEST 5: Regional Demand Summary Valid
# ---------------------------------------------------
def test_regional_summary(tourism_assets):
    regional = tourism_assets["regional"]

    assert len(regional) > 0
    assert "district" in regional.columns

# ---------------------------------------------------
# TEST 6: Linear Regression Model Predictions
# ---------------------------------------------------
def test_linear_regression_predictions(tourism_assets):
    lin_model = tourism_assets["linear"]

    # Ensure it's a LinearRegression instance
    assert isinstance(lin_model, LinearRegression)

    # Get number of expected features
    n_features = lin_model.n_features_in_

    # Create dummy input with correct feature count
    sample_input = np.ones((1, n_features))

    pred = lin_model.predict(sample_input)

    assert isinstance(pred, np.ndarray)
    assert pred.shape == (1,)

Overwriting test_tourism_models.py


In [8]:
!pytest -p no:warnings test_tourism_models.py

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.12.1, typeguard-4.5.1, langsmith-0.7.16
collected 6 items                                                              

test_tourism_models.py ......                                            [100%]

============================== 6 passed in 3.63s ===============================
